In [1]:
"""
This script demonstrates a multi-agent system that uses a sequential workflow
to answer, verify, and refine responses from a search tool.

--- How to Run This Script ---

1. Prerequisites:
   - Python 3.6+
   - pip (Python package installer)

2. Installation:
   Install the required Python libraries by running the following command in your
   terminal:
   pip install requests python-dotenv

3. Configuration:
   a. Create a file named `.env` in the same directory as this script.
   b. Get a Google Maps Geocoding API key from the Google Cloud Console and
      ensure the "Geocoding API" is enabled for your project.
   c. Add your API key to the `.env` file like this:
      GOOGLE_MAPS_API_KEY="YOUR_API_KEY_HERE"

4. Execution:
   Run the script from your terminal:
   python3 multi_agent_system.py

"""

from typing import Dict

# --- Simulated Search Tool ---


def google_search_tool(query: str) -> str:
    """Simulates calling a Google Search tool."""
    print(f"EVENT: Search tool called with query: '{query}'\n")
    # In a real ADK, this would make an actual API call.
    # Here, we return a mock result.
    if "impact of AI on software development" in query.lower():
        return (
            "AI is transforming software development by automating tasks like "
            "code generation, testing, and bug detection. It also helps in "
            "optimizing code and improving developer productivity."
        )
    return "No search results found."


# --- Agent Definitions ---


class Agent:
    """A base class for simple, rule-based agents."""

    def __init__(self, name: str, instructions: str):
        self.name = name
        self.instructions = instructions

    def run(self, state: Dict) -> str:
        """
        Runs the agent. In a real system, this would involve an LLM call.
        Here, we simulate the agent's behavior based on its instructions.
        """
        raise NotImplementedError


class GreeterAgent(Agent):
    """A simple agent that provides a greeting."""

    def run(self, state: Dict) -> str:
        print(f"EVENT: '{self.name}' is running.")
        print(f"Instructions: {self.instructions}\n")
        return f"Hello! I see you want to know about: '{state['query']}'. I can help with that."


class SearchAgent(Agent):
    """An agent that uses a search tool to find information."""

    def run(self, state: Dict) -> str:
        print(f"EVENT: '{self.name}' is running.")
        print(f"Instructions: {self.instructions}\n")
        query = state["query"]
        return google_search_tool(query)


class CritiqueAgent(Agent):
    """An agent that critiques a response based on a set of rules."""

    def run(self, state: Dict) -> str:
        print(f"EVENT: '{self.name}' is running.")
        print(f"Instructions: {self.instructions}")
        print(f"Critiquing the following response: '{state['initial_answer']}'\n")
        # In a real system, an LLM would generate this critique.
        return (
            "The response is a good start, but it is too brief. "
            "Expand on the key areas mentioned, such as code generation and testing. "
            "Also, provide a concluding sentence."
        )


class RefineAgent(Agent):
    """An agent that refines a response based on a critique."""

    def run(self, state: Dict) -> str:
        print(f"EVENT: '{self.name}' is running.")
        print(f"Instructions: {self.instructions}")
        print(f"Refining based on critique: '{state['critique']}'\n")
        # In a real system, an LLM would rewrite the response.
        refined_answer = (
            f"{state['initial_answer']} Specifically, in code generation, AI can "
            "create boilerplate code and algorithms, freeing up developers. "
            "In testing, it can automatically generate test cases and identify "
            "edge cases that might be missed. This leads to more robust and "
            "reliable software."
        )
        return refined_answer


# --- Agent Team / Workflow ---


class SequentialTeam:
    """A team of agents that work in a sequence to accomplish a task."""

    def __init__(self, agents: list):
        self.agents = agents

    def run(self, initial_state: Dict):
        """Runs the sequential workflow."""
        state = initial_state.copy()
        print(f"--- Starting Sequential Team with query: '{state['query']}' ---\n")

        # Run the Greeter Agent first
        greeting = self.agents[0].run(state)
        print(f"GREETING: {greeting}\n")
        print("-" * 20)

        # Run the Search Agent
        initial_answer = self.agents[1].run(state)
        state["initial_answer"] = initial_answer
        print(f"SEARCH RESULT: {initial_answer}\n")
        print("-" * 20)

        # Run the Critique Agent
        critique = self.agents[2].run(state)
        state["critique"] = critique
        print(f"CRITIQUE: {critique}\n")
        print("-" * 20)

        # Run the Refine Agent
        final_answer = self.agents[3].run(state)
        print(f"FINAL ANSWER: {final_answer}\n")
        print("--- Sequential Team Finished ---")


def test_answer_refinement_workflow():
    """Tests the full answer, critique, and refinement workflow."""

    # 1. Define the agents
    greeter_agent = GreeterAgent(
        name="GreeterAgent",
        instructions="You are a friendly assistant that greets the user and confirms their query.",
    )
    search_agent = SearchAgent(
        name="SearchAgent",
        instructions="Use the search tool to find an initial answer to the user's query.",
    )
    critique_agent = CritiqueAgent(
        name="CritiqueAgent",
        instructions="Review the initial answer and provide constructive feedback for improvement.",
    )
    refine_agent = RefineAgent(
        name="RefineAgent",
        instructions="Rewrite the initial answer based on the critique to create a more complete and polished final response.",
    )

    # 2. Create the sequential team
    answer_team = SequentialTeam(
        agents=[greeter_agent, search_agent, critique_agent, refine_agent]
    )

    # 3. Define the initial state with a user query
    query = "What is the impact of AI on software development?"
    initial_state = {"query": query}

    # 4. Run the workflow
    answer_team.run(initial_state)


if __name__ == "__main__":
    test_answer_refinement_workflow()



--- Starting Sequential Team with query: 'What is the impact of AI on software development?' ---

EVENT: 'GreeterAgent' is running.
Instructions: You are a friendly assistant that greets the user and confirms their query.

GREETING: Hello! I see you want to know about: 'What is the impact of AI on software development?'. I can help with that.

--------------------
EVENT: 'SearchAgent' is running.
Instructions: Use the search tool to find an initial answer to the user's query.

EVENT: Search tool called with query: 'What is the impact of AI on software development?'

SEARCH RESULT: No search results found.

--------------------
EVENT: 'CritiqueAgent' is running.
Instructions: Review the initial answer and provide constructive feedback for improvement.
Critiquing the following response: 'No search results found.'

CRITIQUE: The response is a good start, but it is too brief. Expand on the key areas mentioned, such as code generation and testing. Also, provide a concluding sentence.

-----